# Spotify Songs Data Analysis with Python

## DS 220 Project #2: Data Analysis with Pandas

### Introduction

For this project, I chose to analyze a Spotify songs dataset because music streaming is a major part of how people discover, share, and measure the success of music today. Spotify data is useful for analysis because it includes both popularity measures, such as streams and playlist placement, and music characteristics, such as danceability, energy, valence, acousticness, speechiness, and BPM.

The goal of this notebook is to use Python, especially the Pandas library, to clean the dataset, explore patterns, create visualizations, answer data analysis questions, and develop a clear story about what factors may be connected to Spotify success.


## Dataset

The dataset used in this project is named `spotify_songs.csv`. It includes information about popular songs, including:

- song title
- artist
- genre
- release year
- total Spotify streams
- Spotify playlist appearances
- chart appearances
- BPM
- mode
- energy
- danceability
- valence
- acousticness
- speechiness
- popularity

This dataset is helpful because it allows both descriptive analysis and relationship-based analysis. For example, I can identify the most-streamed songs, compare artists and genres, and test whether song traits like danceability or energy are related to popularity.


## Questions This Analysis Will Answer

1. Which songs have the most Spotify streams?
2. Which artists have the highest total streams?
3. Which genres have the highest average popularity?
4. How do streams change by year?
5. Is Spotify playlist placement related to streams?
6. Is danceability related to popularity?
7. Is energy related to popularity?
8. Which song traits are most correlated with popularity?
9. Do major-mode and minor-mode songs differ in average popularity?
10. What final story does the data tell about Spotify success?


## Import Libraries

This project mainly uses Pandas for data analysis. Pandas is used to load the CSV file, clean the dataset, group rows, calculate averages and correlations, sort values, and create tables. Matplotlib is used to create the visualizations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import shutil

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## Load the Data

This notebook reads the file `spotify_songs.csv` directly. In Google Colab, make sure the CSV file is uploaded in the file panel before running this cell.


In [ ]:
df = pd.read_csv("spotify_songs.csv")

print("Rows and columns:", df.shape)
df.head()


## Inspect the Raw Data

Before answering questions, I first check the column names, data types, and missing values. This helps identify whether any cleaning is needed.


In [ ]:
print("Original column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())


## Clean and Prepare the Data

The cleaning process standardizes the column names, removes duplicate rows, converts numeric-looking columns into numeric data types, and creates a new column called `streams_millions`. This makes the stream numbers easier to read.


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

before_duplicates = len(df)
df = df.drop_duplicates()
after_duplicates = len(df)

print("Duplicate rows removed:", before_duplicates - after_duplicates)

for col in df.columns:
    if df[col].dtype == "object":
        cleaned = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        numeric_version = pd.to_numeric(cleaned, errors="coerce")
        if numeric_version.notna().sum() >= len(df) * 0.55:
            df[col] = numeric_version

df["streams_millions"] = df["streams"] / 1_000_000

print("Cleaned columns:")
print(df.columns.tolist())

df.head()


## Exploratory Data Analysis

This section summarizes the numeric columns. It gives a quick overview of the range, average, and spread of the dataset.


In [ ]:
numeric_summary = df.describe().T
numeric_summary


In [ ]:
print("Number of songs:", df["song"].nunique())
print("Number of artists:", df["artist"].nunique())
print("Number of genres:", df["genre"].nunique())
print("Year range:", int(df["year"].min()), "to", int(df["year"].max()))


## Question 1: Which Songs Have the Most Streams?

In [ ]:
top_songs = (
    df[["song", "artist", "year", "streams_millions", "popularity"]]
    .sort_values("streams_millions", ascending=False)
    .head(10)
)

top_songs

In [ ]:
plt.barh(top_songs["song"], top_songs["streams_millions"])
plt.gca().invert_yaxis()
plt.title("Top 10 Songs by Spotify Streams")
plt.xlabel("Streams in Millions")
plt.ylabel("Song")
plt.show()

**Insight:** The highest-streamed songs are not always the newest songs. This suggests that songs can continue gaining streams when they have replay value and strong playlist visibility.

## Question 2: Which Artists Have the Highest Total Streams?

In [ ]:
top_artists = (
    df.groupby("artist", as_index=False)
    .agg(
        total_streams_millions=("streams_millions", "sum"),
        song_count=("song", "count"),
        avg_popularity=("popularity", "mean")
    )
    .sort_values("total_streams_millions", ascending=False)
    .head(10)
)

top_artists

In [ ]:
plt.barh(top_artists["artist"], top_artists["total_streams_millions"])
plt.gca().invert_yaxis()
plt.title("Top 10 Artists by Total Streams")
plt.xlabel("Total Streams in Millions")
plt.ylabel("Artist")
plt.show()

**Insight:** Artists with multiple successful songs have an advantage because their streams add across their catalog.

## Question 3: Which Genres Have the Highest Average Popularity?

In [ ]:
genre_summary = (
    df.groupby("genre")
    .agg(
        avg_popularity=("popularity", "mean"),
        avg_streams_millions=("streams_millions", "mean"),
        song_count=("song", "count")
    )
    .reset_index()
    .sort_values("avg_popularity", ascending=False)
)

top_genres = genre_summary.head(10)
top_genres

In [ ]:
plt.barh(top_genres["genre"], top_genres["avg_popularity"])
plt.gca().invert_yaxis()
plt.title("Top Genres by Average Popularity")
plt.xlabel("Average Popularity")
plt.ylabel("Genre")
plt.show()

**Insight:** Genre averages should be interpreted with song count. A genre with only one very successful song can have a high average.

## Question 4: How Do Streams Change by Year?

In [ ]:
year_summary = (
    df.groupby("year")
    .agg(
        total_streams_millions=("streams_millions", "sum"),
        avg_streams_millions=("streams_millions", "mean"),
        avg_popularity=("popularity", "mean"),
        song_count=("song", "count")
    )
    .reset_index()
    .sort_values("year")
)

year_summary

In [ ]:
plt.plot(year_summary["year"], year_summary["total_streams_millions"], marker="o")
plt.title("Total Spotify Streams by Year")
plt.xlabel("Year")
plt.ylabel("Total Streams in Millions")
plt.show()

**Insight:** Yearly totals depend on how many songs from that year appear in the dataset. A year with several major hits can have a much higher total.

## Question 5: Is Spotify Playlist Placement Related to Streams?

In [ ]:
playlist_stream_corr = df["in_spotify_playlists"].corr(df["streams_millions"])

print("Correlation between Spotify playlists and streams:", round(playlist_stream_corr, 3))

df[["song", "artist", "in_spotify_playlists", "streams_millions"]].sort_values(
    "in_spotify_playlists", ascending=False
).head(10)

In [ ]:
plt.scatter(df["in_spotify_playlists"], df["streams_millions"], alpha=0.75)
plt.title("Spotify Playlists vs. Streams")
plt.xlabel("Number of Spotify Playlists")
plt.ylabel("Streams in Millions")
plt.show()

**Insight:** Playlist placement is important because playlists help songs get discovered and replayed. This supports the idea that platform visibility matters.

## Question 6: Is Danceability Related to Popularity?

In [ ]:
danceability_corr = df["danceability"].corr(df["popularity"])

print("Correlation between danceability and popularity:", round(danceability_corr, 3))

df[["song", "artist", "danceability", "popularity"]].sort_values(
    "danceability", ascending=False
).head(10)

In [ ]:
plt.scatter(df["danceability"], df["popularity"], alpha=0.75)
plt.title("Danceability vs. Popularity")
plt.xlabel("Danceability")
plt.ylabel("Popularity")
plt.show()

**Insight:** Danceability may help describe a song, but it does not fully explain popularity by itself.

## Question 7: Is Energy Related to Popularity?

In [ ]:
energy_corr = df["energy"].corr(df["popularity"])

print("Correlation between energy and popularity:", round(energy_corr, 3))

df[["song", "artist", "energy", "popularity"]].sort_values(
    "energy", ascending=False
).head(10)

In [ ]:
plt.scatter(df["energy"], df["popularity"], alpha=0.75)
plt.title("Energy vs. Popularity")
plt.xlabel("Energy")
plt.ylabel("Popularity")
plt.show()

**Insight:** High energy does not automatically mean high popularity. Some lower-energy songs can still become major hits.

## Question 8: Which Traits Are Most Correlated with Popularity?

In [ ]:
music_traits = [
    "streams_millions",
    "in_spotify_playlists",
    "bpm",
    "energy",
    "danceability",
    "liveness",
    "valence",
    "duration",
    "acousticness",
    "speechiness",
    "popularity"
]

corr_table = df[music_traits].corr().round(2)
corr_table

In [ ]:
popularity_correlations = (
    corr_table["popularity"]
    .drop("popularity")
    .sort_values()
)

plt.barh(popularity_correlations.index, popularity_correlations.values)
plt.axvline(0, linewidth=1)
plt.title("Correlation of Song Traits with Popularity")
plt.xlabel("Correlation with Popularity")
plt.show()

**Insight:** Popularity is connected to several factors rather than one simple variable. Streams and playlist visibility are usually more useful than one audio trait alone.

## Question 9: Do Major-Mode and Minor-Mode Songs Differ?

In [ ]:
mode_summary = (
    df.groupby("mode")
    .agg(
        avg_popularity=("popularity", "mean"),
        avg_streams_millions=("streams_millions", "mean"),
        song_count=("song", "count")
    )
    .reset_index()
)

mode_summary

In [ ]:
plt.bar(mode_summary["mode"], mode_summary["avg_popularity"])
plt.title("Average Popularity by Mode")
plt.xlabel("Mode")
plt.ylabel("Average Popularity")
plt.show()

**Insight:** Mode may affect a song's mood, but it is not a simple rule for predicting whether a song will be popular.

## Final Story and Actionable Insights

The data suggests that Spotify success is not explained by one feature alone. The strongest songs usually combine artist recognition, playlist exposure, replay value, and broad pop appeal. Audio features like danceability and energy are useful for describing songs, but they do not guarantee popularity. For artists, labels, or marketers, the most actionable lesson is to focus on both the song itself and the platform strategy that helps listeners discover it.


## Create HTML Files for GitHub Pages

The next cells create two connected HTML files with renamed filenames:

- `spotify_home.html`
- `spotify_report.html`

These files can be uploaded to GitHub Pages along with the `images` folder.


In [ ]:
output_dir = Path("spotify_site")
images_dir = output_dir / "images"

if output_dir.exists():
    shutil.rmtree(output_dir)

images_dir.mkdir(parents=True)

def save_chart(filename):
    path = images_dir / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    return "images/" + filename

plt.figure(figsize=(8, 5))
plt.barh(top_songs["song"], top_songs["streams_millions"])
plt.gca().invert_yaxis()
plt.title("Top 10 Songs by Spotify Streams")
plt.xlabel("Streams in Millions")
top_songs_chart = save_chart("top_songs.png")

plt.figure(figsize=(8, 5))
plt.barh(top_artists["artist"], top_artists["total_streams_millions"])
plt.gca().invert_yaxis()
plt.title("Top 10 Artists by Total Streams")
plt.xlabel("Total Streams in Millions")
top_artists_chart = save_chart("top_artists.png")

plt.figure(figsize=(8, 5))
plt.barh(top_genres["genre"], top_genres["avg_popularity"])
plt.gca().invert_yaxis()
plt.title("Average Popularity by Genre")
plt.xlabel("Average Popularity")
genre_chart = save_chart("genre_popularity.png")

plt.figure(figsize=(8, 5))
plt.plot(year_summary["year"], year_summary["total_streams_millions"], marker="o")
plt.title("Total Streams by Year")
plt.xlabel("Year")
plt.ylabel("Total Streams in Millions")
year_chart = save_chart("streams_by_year.png")

plt.figure(figsize=(7, 5))
plt.scatter(df["in_spotify_playlists"], df["streams_millions"], alpha=0.75)
plt.title("Spotify Playlists vs. Streams")
plt.xlabel("Spotify Playlists")
plt.ylabel("Streams in Millions")
playlist_chart = save_chart("playlists_vs_streams.png")

plt.figure(figsize=(7, 5))
plt.scatter(df["danceability"], df["popularity"], alpha=0.75)
plt.title("Danceability vs. Popularity")
plt.xlabel("Danceability")
plt.ylabel("Popularity")
dance_chart = save_chart("danceability_popularity.png")

plt.figure(figsize=(7, 5))
plt.scatter(df["energy"], df["popularity"], alpha=0.75)
plt.title("Energy vs. Popularity")
plt.xlabel("Energy")
plt.ylabel("Popularity")
energy_chart = save_chart("energy_popularity.png")

plt.figure(figsize=(8, 5))
plt.barh(popularity_correlations.index, popularity_correlations.values)
plt.axvline(0, linewidth=1)
plt.title("Correlation of Song Traits with Popularity")
plt.xlabel("Correlation")
corr_chart = save_chart("correlations.png")

plt.figure(figsize=(7, 5))
plt.bar(mode_summary["mode"], mode_summary["avg_popularity"])
plt.title("Average Popularity by Mode")
plt.xlabel("Mode")
plt.ylabel("Average Popularity")
mode_chart = save_chart("mode_popularity.png")

print("Charts created in:", images_dir)


In [ ]:
def table_html(dataframe, max_rows=10):
    show = dataframe.head(max_rows).copy()
    for col in show.columns:
        if pd.api.types.is_numeric_dtype(show[col]):
            show[col] = show[col].map(lambda x: f"{x:,.2f}")
    return show.to_html(index=False, border=0, classes="data-table")

site_style = """
<style>
  * { box-sizing: border-box; }
  body {
    margin: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f6f5;
    color: #111827;
    line-height: 1.55;
  }
  header {
    background: #1DB954;
    color: white;
    padding: 70px 10%;
  }
  header h1 {
    margin: 0 0 20px;
    font-size: 44px;
    font-weight: 800;
  }
  header p {
    font-size: 20px;
    margin: 0 0 25px;
    font-weight: 600;
  }
  nav a {
    color: white;
    font-weight: 800;
    font-size: 18px;
    margin-right: 24px;
    text-decoration: underline;
  }
  main {
    max-width: 1120px;
    margin: 0 auto;
    background: white;
    padding: 56px 38px;
    min-height: 100vh;
  }
  .card {
    background: white;
    border-radius: 14px;
    padding: 28px;
    margin-bottom: 28px;
    box-shadow: 0 4px 18px rgba(15, 23, 42, 0.08);
    border: 1px solid #e5e7eb;
  }
  h2 {
    font-size: 30px;
    margin: 0 0 20px;
    font-weight: 800;
  }
  p, li { font-size: 18px; }
  .note {
    border-left: 8px solid #1DB954;
    background: #e9fbf1;
    padding: 20px 24px;
    border-radius: 12px;
    font-size: 18px;
  }
  .table-wrap { overflow-x: auto; }
  .data-table {
    width: 100%;
    border-collapse: collapse;
    margin: 16px 0;
    font-size: 14px;
  }
  .data-table th {
    background: #1DB954;
    color: white;
    padding: 9px;
    text-align: left;
    border: 1px solid #16a34a;
  }
  .data-table td {
    padding: 8px;
    border: 1px solid #d7d7d7;
  }
  .chart-img {
    width: 100%;
    border: 1px solid #d7d7d7;
    border-radius: 10px;
    margin: 14px 0;
    background: white;
  }
  .insight {
    background: #e9fbf1;
    border-left: 5px solid #1DB954;
    border-radius: 8px;
    padding: 14px 16px;
    font-size: 16px;
    margin-top: 14px;
  }
  .button {
    display: inline-block;
    background: #1DB954;
    color: white;
    padding: 12px 18px;
    border-radius: 10px;
    font-weight: 800;
    text-decoration: none;
    margin-top: 8px;
  }
</style>
"""

home_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Spotify Home Page</title>
  {site_style}
</head>
<body>
  <header>
    <h1>Spotify Songs Analysis</h1>
    <p>Python, Pandas, and Matplotlib analysis of a Spotify songs dataset.</p>
    <nav>
      <a href="spotify_home.html">Home</a>
      <a href="spotify_report.html">Analysis</a>
    </nav>
  </header>

  <main>
    <section class="card">
      <h2>Problem Space</h2>
      <p>
        This project studies what makes songs successful on Spotify. The analysis compares streaming totals with playlist placement,
        artist frequency, release timing, and audio features such as danceability, valence, energy, acousticness, liveness, and speechiness.
      </p>
      <a class="button" href="spotify_report.html">View Full Analysis</a>
    </section>

    <section class="card">
      <h2>Dataset Connection</h2>
      <p>
        The Spotify dataset contains both popularity measures and music feature measures. This lets us ask whether highly streamed songs
        are popular because of their sound, their visibility, or both.
      </p>
    </section>

    <section class="card">
      <h2>Questions Addressed</h2>
      <ol>
        <li>Which songs have the most Spotify streams?</li>
        <li>Which artists have the highest total streams?</li>
        <li>Which genres have the highest average popularity?</li>
        <li>How do streams change by year?</li>
        <li>Is playlist placement related to streams?</li>
        <li>Is danceability related to popularity?</li>
        <li>Is energy related to popularity?</li>
        <li>Which traits are most correlated with popularity?</li>
        <li>Do major-mode and minor-mode songs differ?</li>
      </ol>
    </section>

    <section class="card">
      <h2>Expected Insight Story</h2>
      <div class="note">
        The project tests whether audio features alone explain streaming success. The stronger story is that visibility,
        replay value, artist recognition, and song traits work together.
      </div>
    </section>
  </main>
</body>
</html>
"""

report_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Spotify Full Report</title>
  {site_style}
</head>
<body>
  <header>
    <h1>Spotify Songs Data Analysis</h1>
    <p>DS 220 Project 2: Python, pandas, NumPy, and Matplotlib</p>
    <nav>
      <a href="spotify_home.html">Home</a>
      <a href="spotify_report.html">Analysis</a>
    </nav>
  </header>

  <main>
    <section class="card">
      <h2>Dataset</h2>
      <p>
        This report analyzes a Spotify songs dataset with song, artist, genre, year, streams, playlists, audio traits, and popularity.
      </p>
    </section>

    <section class="card">
      <h2>Exploratory Data Analysis</h2>
      <div class="table-wrap">{table_html(numeric_summary.reset_index().rename(columns={"index":"variable"}), 12)}</div>
    </section>

    <section class="card">
      <h2>1. Which songs have the most streams?</h2>
      <div class="table-wrap">{table_html(top_songs, 10)}</div>
      <img class="chart-img" src="{top_songs_chart}" alt="Top songs chart">
      <div class="insight">The top-streamed songs are not always the newest songs, showing that replay value matters.</div>
    </section>

    <section class="card">
      <h2>2. Which artists have the highest total streams?</h2>
      <div class="table-wrap">{table_html(top_artists, 10)}</div>
      <img class="chart-img" src="{top_artists_chart}" alt="Top artists chart">
      <div class="insight">Artists with multiple popular songs have an advantage because their streams add across songs.</div>
    </section>

    <section class="card">
      <h2>3. Which genres have the highest average popularity?</h2>
      <div class="table-wrap">{table_html(genre_summary, 10)}</div>
      <img class="chart-img" src="{genre_chart}" alt="Genre chart">
      <div class="insight">Genre averages should be read with song count because small groups can be affected by one hit song.</div>
    </section>

    <section class="card">
      <h2>4. How do streams change by year?</h2>
      <div class="table-wrap">{table_html(year_summary, 20)}</div>
      <img class="chart-img" src="{year_chart}" alt="Year chart">
      <div class="insight">Yearly stream totals depend on how many major hits are included from each year.</div>
    </section>

    <section class="card">
      <h2>5. Is playlist placement related to streams?</h2>
      <p>Correlation: <strong>{round(playlist_stream_corr, 3)}</strong></p>
      <img class="chart-img" src="{playlist_chart}" alt="Playlist chart">
      <div class="insight">Playlist placement is an important visibility factor for streaming performance.</div>
    </section>

    <section class="card">
      <h2>6. Is danceability related to popularity?</h2>
      <p>Correlation: <strong>{round(danceability_corr, 3)}</strong></p>
      <img class="chart-img" src="{dance_chart}" alt="Danceability chart">
      <div class="insight">Danceability helps describe songs but does not explain popularity alone.</div>
    </section>

    <section class="card">
      <h2>7. Is energy related to popularity?</h2>
      <p>Correlation: <strong>{round(energy_corr, 3)}</strong></p>
      <img class="chart-img" src="{energy_chart}" alt="Energy chart">
      <div class="insight">Energy does not guarantee popularity because lower-energy songs can still be major hits.</div>
    </section>

    <section class="card">
      <h2>8. Which traits are most correlated with popularity?</h2>
      <div class="table-wrap">{table_html(corr_table.reset_index().rename(columns={"index":"feature"}), 12)}</div>
      <img class="chart-img" src="{corr_chart}" alt="Correlation chart">
      <div class="insight">Popularity is connected to multiple variables rather than one single trait.</div>
    </section>

    <section class="card">
      <h2>9. Do major-mode and minor-mode songs differ?</h2>
      <div class="table-wrap">{table_html(mode_summary, 10)}</div>
      <img class="chart-img" src="{mode_chart}" alt="Mode chart">
      <div class="insight">Mode may affect mood, but it is not a simple rule for predicting popularity.</div>
    </section>

    <section class="card">
      <h2>Final Insights</h2>
      <p>
        Spotify success is not explained by one feature alone. The highest-streamed songs usually combine broad pop appeal,
        playlist visibility, artist recognition, and strong replay value.
      </p>
    </section>
  </main>
</body>
</html>
"""

(output_dir / "spotify_home.html").write_text(home_html, encoding="utf-8")
(output_dir / "spotify_report.html").write_text(report_html, encoding="utf-8")

print("Created spotify_home.html and spotify_report.html")


In [ ]:
zip_name = "spotify_project2_site.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_dir.rglob("*"):
        z.write(file, file.relative_to(output_dir))

print("Created:", zip_name)


## Useful Resources

- Pandas documentation: https://pandas.pydata.org/docs/
- Matplotlib documentation: https://matplotlib.org/stable/users/index.html
- GitHub Pages documentation: https://pages.github.com/
